In [1]:
import pandas as pd
import json

In [3]:

license_df = pd.read_csv("../data/license.csv", dtype=str)
installed_df = pd.read_json("../data/installed.json")
altered_df = pd.read_json("../data/altered.json")
inspection_df = pd.read_csv("../data/inspection.csv")


In [4]:

print("LICENSE:", license_df.shape)
print("INSTALLED:", installed_df.shape)
print("ALTERED:", altered_df.shape)
print("INSPECTION:", inspection_df.shape)

LICENSE: (45383, 11)
INSTALLED: (46936, 9)
ALTERED: (31619, 10)
INSPECTION: (143181, 9)


In [5]:
license_df.columns
installed_df.columns
altered_df.columns
inspection_df.columns

Index(['originatingservicerequestnumber', 'InspectionCustomer',
       'ElevatingDevicesNumber', 'InspectionNumber', 'InspectionLocation',
       'InspectionType', 'Earliest_INSPECTION_Date', 'Latest_INSPECTION_Date',
       'InspectionOutcome'],
      dtype='str')

## AND-2 Task 5: Merge 1 — License + Installed

In [9]:
license_df = license_df[
    license_df["LICENSESTATUS"].str.strip().isin(["ACTIVE", "BY REQUEST"])
].copy()

print("Filtered License rows:", len(license_df))

installed_df["Elevating devices number"] = installed_df["Elevating devices number"].astype(str)
installed_df["Elevating devices number"] = installed_df["Elevating devices number"].str.strip()
license_df["ElevatingDevicesNumber"] = license_df["ElevatingDevicesNumber"].str.strip()

Filtered License rows: 43002


Before merging, the key column from the installed dataset was converted to string to match the license dataset. This ensures a valid join operation and prevents type mismatch errors.

In [12]:
print(license_df["ElevatingDevicesNumber"].dtype)
print(installed_df["Elevating devices number"].dtype)

str
str


In [14]:
merged_df = license_df.merge(
    installed_df,
    left_on="ElevatingDevicesNumber",
    right_on="Elevating devices number",
    how="inner"
)

print("License filtered:", len(license_df))
print("Installed:", len(installed_df))
print("Merged:", len(merged_df))

License filtered: 43002
Installed: 46936
Merged: 43002


In [15]:
print(merged_df.columns)

Index(['ElevatingDevicesNumber', 'LocationoftheElevatingDevice',
       'ElevatingDevicesLicenseNumber', 'LICENSESTATUS', 'LICENSEEXPIRYDATE',
       'LICENSEHOLDER', 'LICENSEHOLDERACCOUNTNUMBER', 'LICENSEHOLDERADDRESS',
       'BILLINGCUSTOMER', 'BILLINGADDRESS', 'BILLINGACCOUNT',
       'Elevating devices number', 'Owner Name', 'Owner Address',
       'Owner Account Number', 'Device Class', 'Device Type', 'DeviceStatus',
       'Location of Device', 'under review'],
      dtype='str')


In [18]:
merged_df["license_city"] = merged_df["LocationoftheElevatingDevice"].str.split().str[0]
merged_df["installed_city"] = merged_df["Location of Device"].str.split().str[0]

In [19]:
mismatch = merged_df[
    merged_df["license_city"] != merged_df["installed_city"]
]

print("Mismatched rows:", len(mismatch))

Mismatched rows: 75


In [20]:
filtered_df = merged_df[
    merged_df["license_city"] == merged_df["installed_city"]
].copy()

print("After location filtering:", len(filtered_df))

After location filtering: 42927


### Merge Results

- License (filtered): 43002 rows  
- Installed: 46936 rows  
- After merge: 43002 rows  

The merge retained all filtered license records, indicating that all ElevatingDevicesNumber values in the filtered license dataset exist in the installed dataset.

### Location Comparison

Location consistency between the two datasets was verified by extracting the first token from each location field as a proxy for city.

- License dataset: LocationoftheElevatingDevice  
- Installed dataset: Location of Device  

A match was defined as both extracted values being equal.

Mismatched rows: 75  
Final dataset after filtering: 42927 rows  

Rows with inconsistent location values were removed to ensure alignment between datasets.


### Location Comparison

The installed dataset does not contain a dedicated city column, so the "Location of Device" field was used. Both datasets store location information as strings, so the first token of each string was extracted as a proxy for city.

A match is defined as both extracted values being equal. Rows with mismatched values were removed to ensure consistency between datasets.

In [22]:
merged_df["Device Type"].value_counts()

Device Type
Passenger Elevator      39605
Freight Elevator         1902
LULA Elevator            1166
Observation Elevator      299
Freight Elevator-P         13
Freight Elevator-E          8
Temporary Elevator          4
Sidewalk Elevator           3
Special Installation        1
Material Lift - ATD         1
Name: count, dtype: int64

In [25]:
merged_df["Device Type"] = merged_df["Device Type"].replace({
    "Passenger Elevator": "Passenger",
    "Passenger elevator": "Passenger",
    "Passenger Lift": "Passenger",
    "Freight Elevator": "Freight",
    "Freight lift": "Freight"
})
merged_df["Device Type"] = merged_df["Device Type"].replace({
    "LULA Elevator": "Other",
    "Observation Elevator": "Other",
    "Freight Elevator-P": "Other",
    "Freight Elevator-E": "Other",
    "Temporary Elevator": "Other",
    "Sidewalk Elevator": "Other",
    "Special Installation": "Other",
    "Material Lift - ATD": "Other"
})

In [26]:
merged_df["Device Type"].value_counts()

Device Type
Passenger    39605
Freight       1902
Other         1495
Name: count, dtype: int64

### Category Cleaning

The "Device Type" column contained multiple categories, including some with very low frequency. 

To improve consistency and simplify analysis, major categories such as "Passenger Elevator" and "Freight Elevator" were retained, while less frequent categories were grouped into a single "Other" category.

This reduction minimizes fragmentation and ensures more reliable aggregation in later steps.

The dataset was already relatively clean for the main categories, so the transformation focused on consolidating low-frequency categories rather than correcting inconsistencies.

## AND-2 Task 5: Merge 2 — Adding Alterations


In [27]:
print(altered_df.columns)

Index(['originating service request number', 'Alteration Customer', 'Summary',
       'Elevating Devices Number', 'Inspection number', 'Alteration  Location',
       'Alteration Type', 'Status of Alteration Request',
       'Alteration contractor name', 'Billing Customer'],
      dtype='str')


In [39]:
for col in altered_df.columns:
    print(f"'{col}'")

[col for col in altered_df.columns if "Elevating" in col]

'originating service request number'
'Alteration Customer'
'Summary'
'Elevating Devices Number'
'Inspection number'
'Alteration  Location'
'Alteration Type'
'Status of Alteration Request'
'Alteration contractor name'
'Billing Customer'


['Elevating Devices Number']

In [42]:
# asegurar tipos
filtered_df["ElevatingDevicesNumber"] = filtered_df["ElevatingDevicesNumber"].astype(str).str.strip()
altered_df["Elevating Devices Number"] = altered_df["Elevating Devices Number"].astype(str).str.strip()

# merge
merged2_df = filtered_df.merge(
    altered_df,
    left_on="ElevatingDevicesNumber",
    right_on="Elevating Devices Number",
    how="left"
)


In [44]:
print("Before merge:", len(filtered_df))
print("After merge:", len(merged2_df))

Before merge: 42927
After merge: 52031


In [45]:
alter_counts = merged2_df.groupby("ElevatingDevicesNumber").size()

high_alter = alter_counts[alter_counts >= 5]

print("Elevators with 5+ alterations:", len(high_alter))

Elevators with 5+ alterations: 51


In [46]:
total = filtered_df["ElevatingDevicesNumber"].nunique()
proportion = len(high_alter) / total

print("Proportion:", proportion)

Proportion: 0.0011880634565657978


### Alterations Analysis

A left join was used to merge the alteration dataset, ensuring that elevators without alteration records were retained.

- Elevators with 5 or more alterations: 51  
- Total number of elevators: 42927  
- Proportion of the fleet: ~0.12%  

This indicates that only a very small portion of the fleet undergoes frequent modifications. Most elevators have few or no alteration records, confirming that high-frequency alterations are rare.

The increase in row count after the merge confirms a one-to-many relationship between elevators and alterations, where a single elevator can have multiple alteration records.


## AND-2 Task 5: Merge 3 — INSPECTIONS

In [47]:
unique_elevators = inspection_df["ElevatingDevicesNumber"].nunique()
total_rows = len(inspection_df)

print("Unique elevators:", unique_elevators)
print("Total inspection records:", total_rows)

Unique elevators: 40954
Total inspection records: 143181


### Inspection Relationship

The inspection dataset contains multiple records per elevator, indicating a one-to-many relationship.

Each elevator can have multiple inspection events over time, which means that merging this dataset directly would duplicate rows.

Therefore, a decision must be made to handle this relationship appropriately.


In [48]:
inspection_df["Latest_INSPECTION_Date"] = pd.to_datetime(
    inspection_df["Latest_INSPECTION_Date"],
    errors="coerce"
)

In [49]:
inspection_latest = (
    inspection_df
    .sort_values(by="Latest_INSPECTION_Date", ascending=False)
    .drop_duplicates(subset="ElevatingDevicesNumber", keep="first")
)

In [50]:
print("Before:", len(inspection_df))
print("After:", len(inspection_latest))

Before: 143181
After: 40954


In [52]:
merged2_df["ElevatingDevicesNumber"] = merged2_df["ElevatingDevicesNumber"].astype(str).str.strip()
inspection_latest["ElevatingDevicesNumber"] = inspection_latest["ElevatingDevicesNumber"].astype(str).str.strip()

In [53]:
final_df = merged2_df.merge(
    inspection_latest,
    on="ElevatingDevicesNumber",
    how="left"
)

In [54]:
print("Before merge:", len(merged2_df))
print("After merge:", len(final_df))

Before merge: 52031
After merge: 52031


Before merging, the inspection dataset key was converted to string to match the existing dataset. This ensures consistent joins and prevents type mismatch errors.

In [55]:
final_df.to_csv("../data/merged_elevator_data.csv", index=False)

### Final Merge and Output

After resolving the one-to-many relationship in the inspection dataset, only the most recent inspection per elevator was retained.

The cleaned dataset was then merged using a left join, ensuring no loss of records.

The final dataset contains integrated information from all four sources: license, installed, alterations, and inspections.

The dataset was saved to data/merged_elevator_data.csv for further analysis.